# Notebook 10 — Biomarker Data Verification and Feature Integration

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI  
**Stage:** Biomarker tabular data verification and feature integration  
**No ML modeling is performed in this notebook.**

## Objective
Verify whether PPMI biomarker tabular files provide usable baseline/screening predictors for the primary analytic cohort, then create integrated feature matrices for later model comparison.

## Expected inputs
Place the biomarker ZIP file in one of these folders:

- `MyDrive/PPMI_PD_Progression/data/raw/additional/`
- `MyDrive/PPMI_PD_Progression/data/additional/`

Expected ZIP name:

`PPMI_Biomarker_Tabular_Selected_01Jul2026.zip`

## Expected outputs
Outputs will be saved to:

`MyDrive/PPMI_PD_Progression/outputs/notebook_10_biomarker_integration/`


In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import re
import json
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import joblib

RANDOM_STATE = 42
ID_COL = "PATNO"
TARGET_COL = "rapid_progression_q75"

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
NB08_DIR = PROJECT_DIR / "outputs" / "notebook_08_multimodal_preprocessing"

RAW_DIRS = [
    PROJECT_DIR / "data" / "raw" / "additional",
    PROJECT_DIR / "data" / "additional",
    PROJECT_DIR / "data" / "raw",
]

EXTRACT_DIR = PROJECT_DIR / "data" / "extracted" / "biomarkers_selected"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_10_biomarker_integration"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("NB04_DIR:", NB04_DIR, "| exists:", NB04_DIR.exists())
print("NB08_DIR:", NB08_DIR, "| exists:", NB08_DIR.exists())
print("OUT_DIR:", OUT_DIR)


In [ ]:
# ============================================================
# 03. Locate biomarker ZIP file
# ============================================================

zip_candidates = []

for raw_dir in RAW_DIRS:
    print("Checking:", raw_dir, "| exists:", raw_dir.exists())
    if raw_dir.exists():
        patterns = [
            "*Biomarker*Tabular*Selected*.zip",
            "*biomarker*tabular*selected*.zip",
            "*Biospecimen*.zip",
            "*SAA*.zip",
            "*.zip",
        ]
        for pat in patterns:
            zip_candidates.extend(list(raw_dir.glob(pat)))

zip_candidates = sorted(set(zip_candidates))

print("\nZIP candidates found:")
for z in zip_candidates:
    print("-", z)

if len(zip_candidates) == 0:
    raise FileNotFoundError(
        "No biomarker ZIP file found. Place PPMI_Biomarker_Tabular_Selected_01Jul2026.zip "
        "in data/raw/additional or data/additional."
    )

# Prefer the expected biomarker ZIP name if present.
preferred = [z for z in zip_candidates if "Biomarker" in z.name or "biomarker" in z.name]
BIOMARKER_ZIP = preferred[0] if preferred else zip_candidates[0]

print("\nUsing biomarker ZIP:", BIOMARKER_ZIP)


In [ ]:
# ============================================================
# 04. Extract ZIP and create file inventory
# ============================================================

with zipfile.ZipFile(BIOMARKER_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

all_files = sorted([p for p in EXTRACT_DIR.rglob("*") if p.is_file()])

inventory_rows = []
for p in all_files:
    rows = np.nan
    cols = np.nan
    if p.suffix.lower() == ".csv":
        try:
            header = pd.read_csv(p, nrows=0)
            cols = header.shape[1]
            with open(p, "rb") as f:
                rows = max(sum(1 for _ in f) - 1, 0)
        except Exception:
            pass
    inventory_rows.append({
        "file_name": p.name,
        "relative_path": str(p.relative_to(EXTRACT_DIR)),
        "extension": p.suffix.lower(),
        "size_bytes": p.stat().st_size,
        "n_rows_if_csv": rows,
        "n_columns_if_csv": cols,
    })

file_inventory = pd.DataFrame(inventory_rows)
file_inventory.to_csv(OUT_DIR / "01_biomarker_file_inventory.csv", index=False)

display(file_inventory)


In [ ]:
# ============================================================
# 05. Load reference train/test raw splits
# ============================================================

# Prefer Notebook 08 multimodal raw splits if available; otherwise use Notebook 04 clinical raw splits.
nb08_train = NB08_DIR / "06_multimodal_train_raw_split_before_preprocessing.csv"
nb08_test = NB08_DIR / "07_multimodal_test_raw_split_before_preprocessing.csv"
nb08_types = NB08_DIR / "04_multimodal_feature_type_dictionary.csv"

nb04_train = NB04_DIR / "04_train_raw_split_before_preprocessing.csv"
nb04_test = NB04_DIR / "05_test_raw_split_before_preprocessing.csv"
nb04_types = NB04_DIR / "03_feature_type_dictionary.csv"

if nb08_train.exists() and nb08_test.exists():
    BASE_INPUT_DIR = NB08_DIR
    train_raw = pd.read_csv(nb08_train)
    test_raw = pd.read_csv(nb08_test)
    feature_types_existing = pd.read_csv(nb08_types) if nb08_types.exists() else pd.DataFrame()
    reference_feature_set = "clinical_plus_datscan_from_notebook_08"
else:
    BASE_INPUT_DIR = NB04_DIR
    train_raw = pd.read_csv(nb04_train)
    test_raw = pd.read_csv(nb04_test)
    feature_types_existing = pd.read_csv(nb04_types) if nb04_types.exists() else pd.DataFrame()
    reference_feature_set = "clinical_only_from_notebook_04"

for df_name, df in [("train_raw", train_raw), ("test_raw", test_raw)]:
    if ID_COL not in df.columns or TARGET_COL not in df.columns:
        raise ValueError(f"{df_name} must contain {ID_COL} and {TARGET_COL}.")

train_raw[ID_COL] = train_raw[ID_COL].astype(str)
test_raw[ID_COL] = test_raw[ID_COL].astype(str)

primary_ids = pd.concat([train_raw[[ID_COL]], test_raw[[ID_COL]]], axis=0).drop_duplicates()
primary_ids[ID_COL] = primary_ids[ID_COL].astype(str)
primary_id_set = set(primary_ids[ID_COL])

print("Reference feature set:", reference_feature_set)
print("Train shape:", train_raw.shape)
print("Test shape:", test_raw.shape)
print("Primary analytic cohort n:", len(primary_ids))
print("Train positive rate:", train_raw[TARGET_COL].mean())
print("Test positive rate:", test_raw[TARGET_COL].mean())


In [ ]:
# ============================================================
# 06. Helper functions
# ============================================================

def clean_feature_name(x):
    x = str(x)
    x = x.strip()
    x = re.sub(r"[^A-Za-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    if len(x) == 0:
        x = "unknown"
    return x[:120]

def find_file_contains(keywords, suffix=".csv"):
    keywords_lower = [k.lower() for k in keywords]
    candidates = []
    for p in EXTRACT_DIR.rglob(f"*{suffix}"):
        name_lower = p.name.lower()
        if all(k in name_lower for k in keywords_lower):
            candidates.append(p)
    return sorted(candidates)

def event_rank(series, priority):
    mapping = {event: rank for rank, event in enumerate(priority)}
    return series.astype(str).map(mapping).fillna(len(priority)).astype(int)

def coerce_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

def select_baseline_like_rows(df, event_col, priority):
    tmp = df.copy()
    tmp[event_col] = tmp[event_col].astype(str)
    tmp = tmp[tmp[event_col].isin(priority)].copy()
    tmp["_event_rank"] = event_rank(tmp[event_col], priority)
    return tmp

def safe_display_head(df, n=5):
    display(df.head(n))
    print("Shape:", df.shape)


In [ ]:
# ============================================================
# 07. Build SAA Biospecimen Analysis feature matrix
# ============================================================

saa_files = find_file_contains(["SAA", "Biospecimen", "Analysis", "Results"])
print("SAA biospecimen files:", [p.name for p in saa_files])

saa_feature_matrix = primary_ids.copy()

if len(saa_files) > 0:
    saa_path = saa_files[0]
    saa = pd.read_csv(saa_path, low_memory=False)
    saa[ID_COL] = saa[ID_COL].astype(str)

    # SAA analysis file is strongest at BL; keep BL first, then SC/ST if needed.
    saa_base = select_baseline_like_rows(saa[saa[ID_COL].isin(primary_id_set)], "CLINICAL_EVENT", ["BL", "SC", "ST"])

    # Numeric SAA kinetic summary columns available in this file.
    exclude_cols = {
        "PATNO", "SEX", "COHORT", "CLINICAL_EVENT", "TYPE", "SAAMethod",
        "SAA_Status", "SAA_Type", "PROJECTID", "PI_NAME", "PI_INSTITUTION", "update_stamp"
    }
    candidate_numeric_cols = [c for c in saa_base.columns if c not in exclude_cols and not c.startswith("_")]
    for c in candidate_numeric_cols:
        saa_base[c] = coerce_numeric_series(saa_base[c])

    # Status aggregation: Positive if any positive; Negative if all negative; otherwise missing.
    def status_to_num(x):
        vals = set(str(v).strip().lower() for v in x.dropna())
        if "positive" in vals:
            return 1
        if vals and vals.issubset({"negative"}):
            return 0
        return np.nan

    # Use first available baseline-like event per participant for event audit.
    event_audit = (
        saa_base.sort_values([ID_COL, "_event_rank"])
        .drop_duplicates(ID_COL, keep="first")[[ID_COL, "CLINICAL_EVENT"]]
        .rename(columns={"CLINICAL_EVENT": "saa_selected_event"})
    )

    numeric_agg = saa_base.groupby(ID_COL)[candidate_numeric_cols].mean(numeric_only=True).reset_index()
    status_agg = saa_base.groupby(ID_COL)["SAA_Status"].apply(status_to_num).reset_index(name="saa_SAA_Status_binary")

    numeric_agg = numeric_agg.rename(columns={c: f"saa_{clean_feature_name(c)}" for c in candidate_numeric_cols})

    saa_features = status_agg.merge(numeric_agg, on=ID_COL, how="outer").merge(event_audit, on=ID_COL, how="left")
    saa_feature_matrix = primary_ids.merge(saa_features, on=ID_COL, how="left")

else:
    print("No SAA Biospecimen Analysis Results file found.")

safe_display_head(saa_feature_matrix)


In [ ]:
# ============================================================
# 08. Build SAA kinetic feature matrix
# ============================================================

kin_files = find_file_contains(["Kinetic", "SAA", "Results"])
print("Kinetic SAA files:", [p.name for p in kin_files])

kinetic_feature_matrix = primary_ids.copy()

if len(kin_files) > 0:
    kin_path = kin_files[0]
    kin = pd.read_csv(kin_path, low_memory=False)
    kin[ID_COL] = kin[ID_COL].astype(str)

    # Kinetic file is strongest at SC; keep SC first, then BL.
    kin_base = select_baseline_like_rows(kin[kin[ID_COL].isin(primary_id_set)], "CLINICAL_EVENT", ["SC", "BL", "ST"])

    numeric_cols = ["Fmax", "TTT", "AUC_Fluoro", "Time_To_Max_Slope", "Max_Slope"]
    numeric_cols = [c for c in numeric_cols if c in kin_base.columns]
    for c in numeric_cols:
        kin_base[c] = coerce_numeric_series(kin_base[c])

    # Aggregate replicates/wells per participant.
    kin_numeric = kin_base.groupby(ID_COL)[numeric_cols].mean(numeric_only=True).reset_index()
    kin_numeric = kin_numeric.rename(columns={c: f"saa_kinetic_{clean_feature_name(c)}" for c in numeric_cols})

    kin_event = (
        kin_base.sort_values([ID_COL, "_event_rank"])
        .drop_duplicates(ID_COL, keep="first")[[ID_COL, "CLINICAL_EVENT"]]
        .rename(columns={"CLINICAL_EVENT": "saa_kinetic_selected_event"})
    )

    kinetic_features = kin_numeric.merge(kin_event, on=ID_COL, how="left")
    kinetic_feature_matrix = primary_ids.merge(kinetic_features, on=ID_COL, how="left")

else:
    print("No Kinetic data for SAA Results file found.")

safe_display_head(kinetic_feature_matrix)


In [ ]:
# ============================================================
# 09. Build Current Biospecimen Analysis feature matrix
# ============================================================

current_files = find_file_contains(["Current", "Biospecimen", "Analysis", "Results"])
print("Current Biospecimen files:", [p.name for p in current_files])

current_feature_matrix = primary_ids.copy()
current_candidate_long = pd.DataFrame()

if len(current_files) > 0:
    current_path = current_files[0]

    usecols = [
        "PATNO", "CLINICAL_EVENT", "TYPE", "TESTNAME", "TESTVALUE", "UNITS", "PROJECTID"
    ]

    current = pd.read_csv(current_path, usecols=lambda c: c in usecols, low_memory=False)
    current[ID_COL] = current[ID_COL].astype(str)
    current = current[current[ID_COL].isin(primary_id_set)].copy()

    # Baseline-like events only.
    current = select_baseline_like_rows(current, "CLINICAL_EVENT", ["BL", "SC", "ST"])
    current["TESTVALUE_NUM"] = pd.to_numeric(current["TESTVALUE"], errors="coerce")
    current = current[current["TESTVALUE_NUM"].notna()].copy()

    # Prefer BL, then SC/ST for each participant-test-matrix combination.
    current["TYPE_CLEAN"] = current["TYPE"].map(clean_feature_name)
    current["TEST_CLEAN"] = current["TESTNAME"].map(clean_feature_name)
    current["bio_feature"] = "bio_current_" + current["TYPE_CLEAN"] + "__" + current["TEST_CLEAN"]

    current = current.sort_values([ID_COL, "bio_feature", "_event_rank"])
    current = current.drop_duplicates([ID_COL, "bio_feature"], keep="first")

    # Avoid huge sparse matrices: keep features with at least 50 observed participants.
    coverage = current.groupby("bio_feature")[ID_COL].nunique().reset_index(name="non_missing_n")
    coverage["missing_pct_in_primary_cohort"] = 100 * (1 - coverage["non_missing_n"] / len(primary_ids))
    kept_features = coverage.loc[coverage["non_missing_n"] >= 50, "bio_feature"].tolist()

    current_candidate_long = current[current["bio_feature"].isin(kept_features)].copy()

    if len(current_candidate_long) > 0:
        current_wide = (
            current_candidate_long
            .pivot_table(index=ID_COL, columns="bio_feature", values="TESTVALUE_NUM", aggfunc="mean")
            .reset_index()
        )
        current_feature_matrix = primary_ids.merge(current_wide, on=ID_COL, how="left")
    else:
        print("No Current Biospecimen Analysis features passed minimum coverage threshold.")

    coverage.to_csv(OUT_DIR / "02_current_biospecimen_initial_feature_coverage.csv", index=False)

else:
    print("No Current Biospecimen Analysis Results file found.")

safe_display_head(current_feature_matrix)


In [ ]:
# ============================================================
# 10. Combine biomarker feature matrices and calculate missingness
# ============================================================

biomarker_features = primary_ids.copy()

for mat in [saa_feature_matrix, kinetic_feature_matrix, current_feature_matrix]:
    new_cols = [c for c in mat.columns if c != ID_COL and c not in biomarker_features.columns]
    if new_cols:
        biomarker_features = biomarker_features.merge(mat[[ID_COL] + new_cols], on=ID_COL, how="left")

# Remove audit event columns from modeling candidates but keep them separately.
audit_cols = [c for c in biomarker_features.columns if c.endswith("_selected_event")]
candidate_biomarker_cols = [c for c in biomarker_features.columns if c not in [ID_COL] + audit_cols]

# Ensure numeric.
for c in candidate_biomarker_cols:
    biomarker_features[c] = pd.to_numeric(biomarker_features[c], errors="coerce")

missingness = []
for c in candidate_biomarker_cols:
    missingness.append({
        "feature": c,
        "missing_n": int(biomarker_features[c].isna().sum()),
        "missing_pct": round(100 * biomarker_features[c].isna().mean(), 2),
        "non_missing_n": int(biomarker_features[c].notna().sum()),
        "unique_non_missing_values": int(biomarker_features[c].nunique(dropna=True)),
    })

biomarker_missingness = pd.DataFrame(missingness).sort_values(["missing_pct", "feature"])

recommended_biomarker_features = biomarker_missingness[
    (biomarker_missingness["missing_pct"] <= 30)
    & (biomarker_missingness["unique_non_missing_values"] > 1)
]["feature"].tolist()

biomarker_features.to_csv(OUT_DIR / "03_baseline_biomarker_feature_matrix.csv", index=False)
biomarker_missingness.to_csv(OUT_DIR / "04_biomarker_feature_missingness.csv", index=False)
pd.DataFrame({"recommended_biomarker_feature": recommended_biomarker_features}).to_csv(
    OUT_DIR / "05_recommended_biomarker_features_missing_le_30pct.csv", index=False
)

print("Total candidate biomarker features:", len(candidate_biomarker_cols))
print("Recommended biomarker features missing <=30%:", len(recommended_biomarker_features))
display(biomarker_missingness.head(30))


In [ ]:
# ============================================================
# 11. Biomarker overlap with train/test/overall cohort
# ============================================================

train_ids = set(train_raw[ID_COL].astype(str))
test_ids = set(test_raw[ID_COL].astype(str))

has_any_biomarker = biomarker_features[[ID_COL] + recommended_biomarker_features].copy()
if recommended_biomarker_features:
    has_any_biomarker["has_recommended_biomarker"] = has_any_biomarker[recommended_biomarker_features].notna().any(axis=1)
else:
    has_any_biomarker["has_recommended_biomarker"] = False

overlap_rows = []
for set_name, ids in [("train", train_ids), ("test", test_ids), ("overall", train_ids.union(test_ids))]:
    sub = has_any_biomarker[has_any_biomarker[ID_COL].isin(ids)]
    overlap_rows.append({
        "set": set_name,
        "n": len(sub),
        "n_with_any_recommended_biomarker": int(sub["has_recommended_biomarker"].sum()),
        "pct_with_any_recommended_biomarker": round(100 * sub["has_recommended_biomarker"].mean(), 2) if len(sub) else np.nan,
    })

overlap_summary = pd.DataFrame(overlap_rows)
overlap_summary.to_csv(OUT_DIR / "06_biomarker_overlap_with_primary_cohort.csv", index=False)
display(overlap_summary)


In [ ]:
# ============================================================
# 12. Integrate biomarkers with reference train/test raw splits
# ============================================================

# Use only recommended features for integration.
biomarker_model_matrix = biomarker_features[[ID_COL] + recommended_biomarker_features].copy()

train_bio_raw = train_raw.merge(biomarker_model_matrix, on=ID_COL, how="left", validate="one_to_one")
test_bio_raw = test_raw.merge(biomarker_model_matrix, on=ID_COL, how="left", validate="one_to_one")

print("Train reference shape:", train_raw.shape)
print("Test reference shape:", test_raw.shape)
print("Train with biomarkers shape:", train_bio_raw.shape)
print("Test with biomarkers shape:", test_bio_raw.shape)

# Missingness by split for recommended biomarkers.
split_missing_rows = []
for split_name, split_df in [("train", train_bio_raw), ("test", test_bio_raw), ("combined", pd.concat([train_bio_raw, test_bio_raw], axis=0))]:
    for c in recommended_biomarker_features:
        split_missing_rows.append({
            "set": split_name,
            "feature": c,
            "missing_n": int(split_df[c].isna().sum()),
            "missing_pct": round(100 * split_df[c].isna().mean(), 2),
            "non_missing_n": int(split_df[c].notna().sum()),
        })

split_missing_df = pd.DataFrame(split_missing_rows)
split_missing_df.to_csv(OUT_DIR / "07_biomarker_feature_missingness_by_split.csv", index=False)

display(split_missing_df.head(20))


In [ ]:
# ============================================================
# 13. Define feature types for integrated matrix
# ============================================================

existing_predictors = [c for c in train_raw.columns if c not in [ID_COL, TARGET_COL]]
biomarker_predictors = recommended_biomarker_features

# Existing feature types.
if feature_types_existing is not None and feature_types_existing.shape[0] > 0:
    existing_types = feature_types_existing.copy()
    if "predictor" not in existing_types.columns:
        existing_types = pd.DataFrame({"predictor": existing_predictors, "feature_type": "continuous"})
    if "source" not in existing_types.columns:
        existing_types["source"] = reference_feature_set
    existing_types = existing_types[existing_types["predictor"].isin(existing_predictors)]
else:
    existing_types = pd.DataFrame({
        "predictor": existing_predictors,
        "feature_type": "continuous",
        "source": reference_feature_set,
    })

# Any untyped existing predictors default to continuous.
typed_existing = set(existing_types["predictor"])
untyped_existing = [c for c in existing_predictors if c not in typed_existing]
if untyped_existing:
    existing_types = pd.concat([
        existing_types,
        pd.DataFrame({
            "predictor": untyped_existing,
            "feature_type": "continuous",
            "source": reference_feature_set + "_untyped_default_continuous",
        })
    ], ignore_index=True)

biomarker_types = pd.DataFrame({
    "predictor": biomarker_predictors,
    "feature_type": "binary",  # corrected below for non-status features
    "source": "biomarker_baseline_or_screening",
})

if len(biomarker_types) > 0:
    biomarker_types.loc[~biomarker_types["predictor"].str.contains("Status_binary", case=False, regex=False), "feature_type"] = "continuous"

integrated_feature_types = pd.concat([existing_types, biomarker_types], ignore_index=True)
integrated_feature_types = integrated_feature_types.drop_duplicates(subset=["predictor"], keep="last")

# Keep only predictors actually in both train and test.
present_cols = set(train_bio_raw.columns).intersection(set(test_bio_raw.columns))
integrated_feature_types = integrated_feature_types[integrated_feature_types["predictor"].isin(present_cols)].copy()

integrated_feature_types.to_csv(OUT_DIR / "08_integrated_feature_type_dictionary.csv", index=False)
display(integrated_feature_types.tail(30))

print("Existing predictors:", len(existing_predictors))
print("Recommended biomarker predictors:", len(biomarker_predictors))
print("Integrated raw predictors:", integrated_feature_types.shape[0])


In [ ]:
# ============================================================
# 14. Train/test preprocessing for integrated biomarker matrix
# ============================================================

predictor_cols = integrated_feature_types["predictor"].tolist()

X_train_raw = train_bio_raw[predictor_cols].copy()
X_test_raw = test_bio_raw[predictor_cols].copy()

y_train = train_bio_raw[TARGET_COL].astype(int).copy()
y_test = test_bio_raw[TARGET_COL].astype(int).copy()
ids_train = train_bio_raw[ID_COL].astype(str).copy()
ids_test = test_bio_raw[ID_COL].astype(str).copy()

# Remove constant predictors using training only.
unique_counts = X_train_raw.nunique(dropna=True)
constant_cols = unique_counts[unique_counts <= 1].index.tolist()

X_train_reduced = X_train_raw.drop(columns=constant_cols)
X_test_reduced = X_test_raw.drop(columns=constant_cols)

pd.DataFrame({"dropped_constant_predictor": constant_cols}).to_csv(
    OUT_DIR / "09_dropped_constant_predictors_integrated_biomarker.csv", index=False
)

feature_types_reduced = integrated_feature_types[~integrated_feature_types["predictor"].isin(constant_cols)].copy()

continuous_features = feature_types_reduced.loc[feature_types_reduced["feature_type"] == "continuous", "predictor"].tolist()
binary_features = feature_types_reduced.loc[feature_types_reduced["feature_type"] == "binary", "predictor"].tolist()
categorical_features = feature_types_reduced.loc[feature_types_reduced["feature_type"] == "categorical", "predictor"].tolist()

# Any remaining unrecognized types default to continuous.
assigned = set(continuous_features + binary_features + categorical_features)
unassigned = [c for c in X_train_reduced.columns if c not in assigned]
if unassigned:
    continuous_features += unassigned

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []

if continuous_features:
    transformers.append((
        "continuous",
        Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        continuous_features
    ))

if binary_features:
    transformers.append((
        "binary",
        Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ]),
        binary_features
    ))

if categorical_features:
    transformers.append((
        "categorical",
        Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot)
        ]),
        categorical_features
    ))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

X_train_processed = preprocessor.fit_transform(X_train_reduced)
X_test_processed = preprocessor.transform(X_test_reduced)

try:
    processed_feature_names = preprocessor.get_feature_names_out().tolist()
except Exception:
    processed_feature_names = []
    processed_feature_names += [f"continuous__{c}" for c in continuous_features]
    processed_feature_names += [f"binary__{c}" for c in binary_features]
    if categorical_features:
        processed_feature_names += [f"categorical__{c}" for c in categorical_features]

X_train_processed_df = pd.DataFrame(X_train_processed, columns=processed_feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=processed_feature_names)

print("X_train reduced:", X_train_reduced.shape)
print("X_test reduced:", X_test_reduced.shape)
print("Processed train:", X_train_processed_df.shape)
print("Processed test:", X_test_processed_df.shape)
print("Missing after preprocessing train:", int(X_train_processed_df.isna().sum().sum()))
print("Missing after preprocessing test:", int(X_test_processed_df.isna().sum().sum()))


In [ ]:
# ============================================================
# 15. Save integrated raw and processed matrices
# ============================================================

train_integrated_raw_out = pd.concat([
    ids_train.reset_index(drop=True).rename(ID_COL),
    y_train.reset_index(drop=True).rename(TARGET_COL),
    X_train_reduced.reset_index(drop=True),
], axis=1)

test_integrated_raw_out = pd.concat([
    ids_test.reset_index(drop=True).rename(ID_COL),
    y_test.reset_index(drop=True).rename(TARGET_COL),
    X_test_reduced.reset_index(drop=True),
], axis=1)

train_integrated_processed_out = pd.concat([
    ids_train.reset_index(drop=True).rename(ID_COL),
    y_train.reset_index(drop=True).rename(TARGET_COL),
    X_train_processed_df.reset_index(drop=True),
], axis=1)

test_integrated_processed_out = pd.concat([
    ids_test.reset_index(drop=True).rename(ID_COL),
    y_test.reset_index(drop=True).rename(TARGET_COL),
    X_test_processed_df.reset_index(drop=True),
], axis=1)

train_integrated_raw_out.to_csv(OUT_DIR / "10_biomarker_integrated_train_raw_split_before_preprocessing.csv", index=False)
test_integrated_raw_out.to_csv(OUT_DIR / "11_biomarker_integrated_test_raw_split_before_preprocessing.csv", index=False)
train_integrated_processed_out.to_csv(OUT_DIR / "12_biomarker_integrated_train_processed_matrix.csv", index=False)
test_integrated_processed_out.to_csv(OUT_DIR / "13_biomarker_integrated_test_processed_matrix.csv", index=False)

pd.DataFrame({"processed_feature_name": processed_feature_names}).to_csv(
    OUT_DIR / "14_biomarker_integrated_processed_feature_names.csv", index=False
)

joblib.dump(preprocessor, OUT_DIR / "15_fitted_biomarker_integrated_preprocessing_pipeline.joblib")

manifest = pd.DataFrame([
    {
        "feature_set": reference_feature_set,
        "source_folder": str(BASE_INPUT_DIR),
        "train_n": train_raw.shape[0],
        "test_n": test_raw.shape[0],
        "raw_predictor_count": len(existing_predictors),
        "processed_feature_count": np.nan,
    },
    {
        "feature_set": reference_feature_set + "_plus_biomarkers",
        "source_folder": str(OUT_DIR),
        "train_n": train_integrated_processed_out.shape[0],
        "test_n": test_integrated_processed_out.shape[0],
        "raw_predictor_count": X_train_reduced.shape[1],
        "processed_feature_count": len(processed_feature_names),
    }
])

manifest.to_csv(OUT_DIR / "16_feature_set_manifest_with_biomarkers.csv", index=False)
display(manifest)

# Sensitivity feature list without baseline_NP3TOT.
processed_without_np3 = [c for c in processed_feature_names if "baseline_NP3TOT" not in c]
pd.DataFrame({"processed_feature_name": processed_without_np3}).to_csv(
    OUT_DIR / "17_biomarker_integrated_processed_feature_names_without_baseline_NP3TOT.csv", index=False
)


In [ ]:
# ============================================================
# 16. Quality Control Checklist
# ============================================================

qc_rows = []

def add_qc(item, status, details):
    qc_rows.append({"qc_item": item, "status": status, "details": details})

add_qc(
    "Biomarker ZIP located and extracted",
    "PASS" if len(all_files) > 0 else "FAIL",
    f"Extracted files: {len(all_files)}"
)

add_qc(
    "Reference train/test splits loaded",
    "PASS" if train_raw.shape[0] > 0 and test_raw.shape[0] > 0 else "FAIL",
    f"reference={reference_feature_set}; train={train_raw.shape}; test={test_raw.shape}"
)

add_qc(
    "Primary participant IDs unique",
    "PASS" if not primary_ids[ID_COL].duplicated().any() else "FAIL",
    f"primary_n={len(primary_ids)}"
)

add_qc(
    "Recommended biomarker features identified",
    "PASS" if len(recommended_biomarker_features) > 0 else "CHECK",
    f"recommended_biomarker_features={len(recommended_biomarker_features)}"
)

add_qc(
    "Train/test split preserved",
    "PASS" if train_integrated_raw_out.shape[0] == train_raw.shape[0] and test_integrated_raw_out.shape[0] == test_raw.shape[0] else "FAIL",
    f"train={train_integrated_raw_out.shape[0]}; test={test_integrated_raw_out.shape[0]}"
)

add_qc(
    "No train/test ID overlap",
    "PASS" if len(set(ids_train).intersection(set(ids_test))) == 0 else "FAIL",
    f"overlap_n={len(set(ids_train).intersection(set(ids_test)))}"
)

add_qc(
    "Preprocessor fit on training data only",
    "PASS",
    "fit_transform on training set; transform on test set."
)

add_qc(
    "No missing values after preprocessing",
    "PASS" if X_train_processed_df.isna().sum().sum() == 0 and X_test_processed_df.isna().sum().sum() == 0 else "FAIL",
    f"train_missing={int(X_train_processed_df.isna().sum().sum())}; test_missing={int(X_test_processed_df.isna().sum().sum())}"
)

add_qc(
    "No ML modeling performed",
    "PASS",
    "This notebook performs biomarker verification, integration, and preprocessing only."
)

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "18_quality_control_checklist.csv", index=False)
display(qc)

if (qc["status"] == "FAIL").any():
    raise RuntimeError("One or more QC checks failed. Review 18_quality_control_checklist.csv before proceeding.")


In [ ]:
# ============================================================
# 17. Summary report
# ============================================================

summary_lines = [
    "Notebook 10 — Biomarker Data Verification and Feature Integration",
    "=" * 72,
    f"Reference feature set: {reference_feature_set}",
    f"Primary analytic cohort n: {len(primary_ids)}",
    f"Train n: {train_raw.shape[0]}",
    f"Test n: {test_raw.shape[0]}",
    "",
    "Biomarker files:",
    f"- Extracted files: {len(all_files)}",
    f"- Candidate biomarker features: {len(candidate_biomarker_cols)}",
    f"- Recommended biomarker features missing <=30%: {len(recommended_biomarker_features)}",
    "",
    "Integrated feature matrix:",
    f"- Raw predictors after constants removed: {X_train_reduced.shape[1]}",
    f"- Processed features: {len(processed_feature_names)}",
    f"- Train processed shape: {X_train_processed_df.shape}",
    f"- Test processed shape: {X_test_processed_df.shape}",
    "",
    "Quality control:",
    f"- QC PASS count: {(qc['status'] == 'PASS').sum()}",
    f"- QC CHECK count: {(qc['status'] == 'CHECK').sum()}",
    f"- QC FAIL count: {(qc['status'] == 'FAIL').sum()}",
    "",
    "Next step:",
    "Notebook 11 should compare reference models against biomarker-integrated models.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUT_DIR / "19_notebook_10_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)
